# 🟡 Interview: Mixed Precision Training Step

fp16 forward → scaled loss → backward → unscale grads → fp32 update

In [ ]:
import torch


In [ ]:
# ✅ INTERVIEW

def mixed_precision_step(model, optimizer, loss_fn, x, y, loss_scale=1024.0):
    """
    手写混合精度训练步骤 —— 面试版。

    面试考点:
    1. 为什么需要混合精度？fp16 快但精度低，fp32 精确但慢
    2. Loss scaling 的原因：fp16 梯度经常下溢到 0
    3. 主权重 (master weights) 必须是 fp32，否则小的参数更新会被舍入
    4. 步骤顺序：half → forward → loss(fp32) → scaled backward → unscale → float → step

    参数: model, optimizer, loss_fn, x, y, loss_scale=1024.0
    返回: float (未缩放的 loss)
    """
    # ---- Step 1: 模型转 fp16 ----
    model.half()
    x_fp16 = x.half()

    # ---- Step 2: fp16 前向 ----
    output = model(x_fp16)

    # ---- Step 3: fp32 loss ----
    # loss 计算必须在 fp32 下，否则累加误差大
    loss = loss_fn(output.float(), y)
    loss_val = loss.item()

    # ---- Step 4: 缩放 + 反向 ----
    optimizer.zero_grad()
    # 乘以 scale 再 backward，让梯度变大，避免 fp16 下溢
    (loss * loss_scale).backward()

    # ---- Step 5: 反缩放梯度 ----
    # 每个参数的梯度都除以 loss_scale 恢复正确量级
    # 同时转为 fp32（参数更新用 fp32）
    for p in model.parameters():
        if p.grad is not None:
            p.grad.data = p.grad.data.float() / loss_scale

    # ---- Step 6: fp32 更新 ----
    model.float()  # 必须在 optimizer.step() 之前恢复 fp32
    optimizer.step()

    return loss_val

In [ ]:
# Verify: run one step on a small Linear model, print loss and confirm weights updated
import torch.nn as nn

torch.manual_seed(42)
model = nn.Linear(8, 4)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
loss_fn = nn.MSELoss()

x = torch.randn(4, 8)
y = torch.randn(4, 4)

weights_before = model.weight.data.clone()

loss_val = mixed_precision_step(model, optimizer, loss_fn, x, y)

print("Loss:", loss_val)
print("Weights updated:", not torch.allclose(model.weight.data, weights_before))
print("Model dtype after step:", model.weight.dtype)

In [ ]:
from torch_judge import check
check("mixed_precision")

## 📝 核心思路总结

1. **混合精度的本质**：前向/反向用 fp16（快 + 省内存），参数更新用 fp32（精确），兼顾速度和数值稳定性。
2. **Loss Scaling 是关键**：fp16 最小正数约 6e-8，小梯度会下溢为 0；乘以 1024 再 backward，之后再除以 1024 恢复。
3. **主权重必须 fp32**：SGD 的更新量 `lr * grad` 可能比 fp16 的精度还小，必须在 fp32 下累加。
4. **步骤顺序不能乱**：half → forward → loss(float) → scaled backward → unscale(grad.float()) → float → step。